In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Mini-GPT 训练与生成演示\n",
    "本Notebook演示从零训练Mini-GPT并生成文本的完整流程"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "import sys\n",
    "sys.path.append('..')\n",
    "\n",
    "from config.config import ModelConfig, TrainingConfig, PathConfig\n",
    "from src.model import MiniGPT\n",
    "from src.dataset import CharDataset, get_batch\n",
    "from src.trainer import Trainer\n",
    "from src.generate import Generator\n",
    "from src.utils import set_seed, get_device"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 配置\n",
    "device = get_device()\n",
    "set_seed(42)\n",
    "\n",
    "model_config = ModelConfig()\n",
    "training_config = TrainingConfig()\n",
    "path_config = PathConfig()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 1. 加载数据\n",
    "with open(path_config.data_path, 'r', encoding='utf-8') as f:\n",
    "    text = f.read()\n",
    "\n",
    "dataset = CharDataset(text, model_config.block_size)\n",
    "model_config.vocab_size = dataset.vocab_size\n",
    "print(f\"数据集大小: {len(dataset)} 样本\")\n",
    "print(f\"词典大小: {dataset.vocab_size}\")\n",
    "print(f\"前100个字符: {text[:100]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 2. 创建模型\n",
    "model = MiniGPT(model_config).to(device)\n",
    "print(f\"模型参数量: {model.get_num_params():,}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 3. 查看一个batch示例\n",
    "x, y = get_batch(dataset, training_config.batch_size, model_config.block_size, device)\n",
    "print(f\"输入形状: {x.shape}\")\n",
    "print(f\"目标形状: {y.shape}\")\n",
    "print(f\"输入示例: {dataset.decode(x[0][:20])}\")\n",
    "print(f\"目标示例: {dataset.decode(y[0][:20])}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 4. 训练（快速演示用少量步数）\n",
    "trainer = Trainer(model, dataset, model_config, training_config, path_config, device)\n",
    "print(\"开始训练...\")\n",
    "trainer.train()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 5. 生成文本\n",
    "generator = Generator(model, dataset.stoi, dataset.itos, device)\n",
    "prompt = \"To be\"\n",
    "print(f\"提示词: {prompt}\")\n",
    "print(\"=\" * 50)\n",
    "generated = generator.generate(prompt)\n",
    "print(generated)\n",
    "print(\"=\" * 50)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}